# Tabular Time Series Related API Demo with NOAA Weather Data

Copyright (c) Microsoft Corporation. All rights reserved. <br>
Licensed under the MIT License.

In this notebook, you will learn how to use the Tabular Time Series related API to filter the data by time windows for sample data uploaded to Azure blob storage. 

The detailed APIs to be demoed in this script are:
- Create Tabular Dataset instance
- Assign fine timestamp column and coarse timestamp column for Tabular Dataset to activate Time Series related APIs
- Clear fine timestamp column and coarse timestamp column
- Filter in data before a specific time
- Filter in data after a specific time
- Filter in data in a specific time range
- Filter in data for recent time range

Besides above APIs, you'll also see:
- Create and load a Workspace
- Load National Oceanic & Atmospheric (NOAA) weather data into Azure blob storage
- Create and register NOAA weather data as a Tabular dataset
- Re-load Tabular Dataset from your Workspace

## Import Dependencies

If you are using an Azure Machine Learning Notebook VM, you are all set. Otherwise, run the cells below to install the Azure Machine Learning Python SDK and create an Azure ML Workspace that's required for this demo.

## Prepare Environment

Print out your version of the Azure ML Python SDK. Version 1.0.60 or above is required for TabularDataset with timeseries attribute. 

In [1]:
import azureml.core
azureml.data.__version__

'1.0.60'

## Import Packages

In [2]:
# import packages
import os

import pandas as pd
import azureml.core
import azureml.data

from calendar import monthrange
from datetime import datetime, timedelta

from azureml.core import Dataset, Datastore, Workspace, Run
from azureml.data.azure_storage_datastore import AzureFileDatastore, AzureBlobDatastore
from azureml.data.dataset_factory import TabularDatasetFactory
from azureml.opendatasets import NoaaIsdWeather

## Set up Configuraton and Create Azure ML Workspace

If you are using an Azure Machine Learning Notebook VM, you are all set. Otherwise, go through the [configuration notebook](https://github.com/Azure/MachineLearningNotebooks/blob/master/configuration.ipynb) first if you haven't already to establish your connection to the Azure ML Workspace.

In [3]:
ws     = Workspace.from_config()
dstore = ws.get_default_datastore()

print(ws.name, ws.resource_group, ws.location, ws.subscription_id, dstore.name, sep = '\n')

azure-ml
copetersrg
southcentralus
60582a10-b9fd-49f1-a546-c4194134bba8
workspaceblobstore


<font color=red>Remove below codes and eanble above codes before releasing it.</font>

## Load Data to Blob Storage

This demo uses public NOAA weather data. You can replace this data with your own. The first cell below creates a Pandas Dataframe object with the first 6 months of 2019 NOAA weather data. The last cell saves the data to a CSV file and uploads the CSV file to Azure blob storage to the location specified in the datapath variable. Currently, the Dataset class only reads uploaded files from blob storage. 

**NOTE:** the below code loads in NOAA ISD weather data from all of 2019. This will take ~15 minutes. To reduce the size of data, we will only keep specific rows with a given stationName.

In [ ]:
%%time 

target_years = [2019]

for year in target_years:
    for month in range(1, 12+1):
        path = 'data/{}/{:02d}/'.format(year, month)
        
        try:  
            start = datetime(year, month, 1)
            end   = datetime(year, month, monthrange(year, month)[1]) + timedelta(days=1)
            isd   = NoaaIsdWeather(start, end).to_pandas_dataframe()
            isd   = isd[isd['stationName'].str.contains('FLORIDA', regex=True, na=False)]
            
            os.makedirs(path, exist_ok=True)
            isd.to_parquet(path + 'data.parquet')
        except Exception as e:
            print('Month {} in year {} likely has no data.\n'.format(month, year))
            print('Exception: {}'.format(e))

Upload data to blob storage.

In [ ]:
dstore.upload('data', 'noaa-isd-data', overwrite=True, show_progress=True)

## Create & Register Tabular Dataset with time-series trait from Blob

The API on Tabular datasets with time-series trait is specially designed to handle Tabular time-series data and time related operations more efficiently. By registering your time-series dataset, you are publishing your dataset to your workspace so that it is accessible to anyone with the same subscription id. 

Create Tabular Dataset instance from blob storage datapath

In [4]:
datastore_path = [(dstore, 'noaa-isd-data/*/*/data.parquet')]
dataset        = Dataset.Tabular.from_parquet_files(path=datastore_path, partition_format = 'noaa-isd-data/{coarse_time:yyyy/MM}/data.parquet')

Assign fine timestamp column for Tabular Dataset to activate Time Series related APIs. The column to be assigned should be a Date type, otherwise the assigning will fail.

In [5]:
tsd = dataset.with_timestamp_columns(fine_grain_timestamp='datetime', coarse_grain_timestamp='coarse_time')

You can convert the entire dataset to a Pandas dataframe, or filter on the timestamp column before converting. Datasets are lazily evaluated.

In [6]:
df  = tsd.to_pandas_dataframe()
df.head(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__,coarse_time
0,722108,12894,2019-01-01 00:53:00,26.536,-81.755,9.0,150.0,3.1,22.8,1020.4,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161811,2019-01-01
1,722108,12894,2019-01-01 01:53:00,26.536,-81.755,9.0,130.0,2.1,21.7,1020.3,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161812,2019-01-01
2,722108,12894,2019-01-01 02:53:00,26.536,-81.755,9.0,100.0,2.6,20.0,1020.6,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161813,2019-01-01
3,722108,12894,2019-01-01 03:53:00,26.536,-81.755,9.0,110.0,1.5,18.9,1020.5,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161814,2019-01-01
4,722108,12894,2019-01-01 04:53:00,26.536,-81.755,9.0,0.0,0.0,18.9,1020.6,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161815,2019-01-01


Register the dataset for easy access from anywhere in Azure ML and keep track of versions, lineage. 

In [7]:
# register dataset to Workspace
registered_ds = tsd.register(ws, 'test-data', create_new_version=True, description='Data for Tabular Dataset with time-series trait demo.', tags={ 'type': 'TabularDataset' })

## Reload the Dataset from Workspace

In [9]:
# get dataset by dataset name
tsd = Dataset.get_by_name(ws, name='test-data')
tsd.to_pandas_dataframe().head(5)

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__,coarse_time
0,722108,12894,2019-01-01 00:53:00,26.536,-81.755,9.0,150.0,3.1,22.8,1020.4,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161811,2019-01-01
1,722108,12894,2019-01-01 01:53:00,26.536,-81.755,9.0,130.0,2.1,21.7,1020.3,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161812,2019-01-01
2,722108,12894,2019-01-01 02:53:00,26.536,-81.755,9.0,100.0,2.6,20.0,1020.6,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161813,2019-01-01
3,722108,12894,2019-01-01 03:53:00,26.536,-81.755,9.0,110.0,1.5,18.9,1020.5,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161814,2019-01-01
4,722108,12894,2019-01-01 04:53:00,26.536,-81.755,9.0,0.0,0.0,18.9,1020.6,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,2161815,2019-01-01


## Filter Data by Time Windows

Once your data has been loaded into the notebook, you can query by time using the time_before(), time_after(), time_between(), and time_recent() functions. You can also choose to drop or keep certain columns.  

### Before Time Input

In [10]:
# select data that occurs before a specified date
tsd2 = tsd.time_before(datetime(2019, 6, 12))
tsd2.to_pandas_dataframe().tail(5)

/anaconda/envs/azureml_py36/lib/python3.6/site-packages/azureml/dataprep/api/dataflow.py:641: UserWarning: Inconsistent or mixed schemas detected across partitions. Using alternate reader.
  warnings.warn('Inconsistent or mixed schemas detected across partitions. Using alternate reader.')


,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__,coarse_time
20566,722108,12894,2019-06-11 19:53:00,26.536,-81.755,9.0,230.0,7.7,35.0,1014.4,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,11,1.0,7576889,2019-06-01
20567,722108,12894,2019-06-11 20:53:00,26.536,-81.755,9.0,240.0,6.2,33.9,1014.4,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,11,1.0,7576890,2019-06-01
20568,722108,12894,2019-06-11 21:53:00,26.536,-81.755,9.0,240.0,6.2,33.9,1014.7,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,11,1.0,7576891,2019-06-01
20569,722108,12894,2019-06-11 22:53:00,26.536,-81.755,9.0,240.0,5.7,32.8,1014.4,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,11,1.0,7576892,2019-06-01
20570,722108,12894,2019-06-11 23:53:00,26.536,-81.755,9.0,230.0,4.6,32.2,1014.6,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,11,1.0,7576893,2019-06-01


## After Time Input

In [11]:
# select data that occurs after a specified date
tsd2 = tsd.time_after(datetime(2019, 6, 1))
tsd2.to_pandas_dataframe().tail(5)

/anaconda/envs/azureml_py36/lib/python3.6/site-packages/azureml/dataprep/api/dataflow.py:641: UserWarning: Inconsistent or mixed schemas detected across partitions. Using alternate reader.
  warnings.warn('Inconsistent or mixed schemas detected across partitions. Using alternate reader.')


,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__,coarse_time
8415,865450,99999,2019-09-02 14:00:00,-34.067,-56.238,68.0,330.0,3.6,12.2,1018.3,...,NaN,NaN,FLORIDA,UY,865450-99999,2019,2,1.0,635079,2019-09-01
8416,865450,99999,2019-09-02 15:00:00,-34.067,-56.238,68.0,300.0,5.1,13.6,1018.0,...,NaN,NaN,FLORIDA,UY,865450-99999,2019,2,1.0,635080,2019-09-01
8417,865450,99999,2019-09-02 16:00:00,-34.067,-56.238,68.0,300.0,5.7,15.2,1017.4,...,NaN,NaN,FLORIDA,UY,865450-99999,2019,2,1.0,635081,2019-09-01
8418,865450,99999,2019-09-02 17:00:00,-34.067,-56.238,68.0,270.0,6.2,15.6,1017.0,...,NaN,NaN,FLORIDA,UY,865450-99999,2019,2,1.0,635082,2019-09-01
8419,865450,99999,2019-09-02 18:00:00,-34.067,-56.238,68.0,270.0,5.7,15.5,1016.3,...,9999.0,NaN,FLORIDA,UY,865450-99999,2019,2,1.0,635083,2019-09-01


## Before & After Time Inputs

You can chain time functions together.

In [15]:
# select data that occurs within a given time range
tsd2 = tsd.time_after(datetime(2019, 7, 31)).time_before(datetime(2019, 8, 18))
tsd2.to_pandas_dataframe().head(5)

/anaconda/envs/azureml_py36/lib/python3.6/site-packages/azureml/dataprep/api/dataflow.py:641: UserWarning: Inconsistent or mixed schemas detected across partitions. Using alternate reader.
  warnings.warn('Inconsistent or mixed schemas detected across partitions. Using alternate reader.')


,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__,coarse_time
0,722108,12894,2019-08-01 00:53:00,26.536,-81.755,9.0,40.0,3.1,23.9,1016.8,...,10.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,1677293,2019-08-01
1,722108,12894,2019-08-01 01:53:00,26.536,-81.755,9.0,20.0,3.6,23.3,1016.6,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,1677294,2019-08-01
2,722108,12894,2019-08-01 02:53:00,26.536,-81.755,9.0,10.0,3.6,23.9,1017.6,...,13.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,1677295,2019-08-01
3,722108,12894,2019-08-01 03:53:00,26.536,-81.755,9.0,40.0,2.6,23.9,1017.4,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,1677296,2019-08-01
4,722108,12894,2019-08-01 04:53:00,26.536,-81.755,9.0,40.0,3.1,23.3,1016.6,...,0.0,NaN,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,1.0,1677297,2019-08-01


### Time Range Input

In [19]:
# another way to select data that occurs within a given time range
tsd2 = tsd.time_between(start_time=datetime(2019, 1, 31, 23, 59, 59), end_time=datetime(2019, 2, 7))
tsd2.to_pandas_dataframe().head(5)

/anaconda/envs/azureml_py36/lib/python3.6/site-packages/azureml/dataprep/api/dataflow.py:641: UserWarning: Inconsistent or mixed schemas detected across partitions. Using alternate reader.
  warnings.warn('Inconsistent or mixed schemas detected across partitions. Using alternate reader.')


,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__,coarse_time
0,854880,99999,2019-02-01 00:00:00,-29.917,-71.2,146.0,290.0,4.6,17.3,1012.2,...,0.0,NaN,LA FLORIDA,CI,854880-99999,2019,1,1.0,1987428,2019-02-01
1,854880,99999,2019-02-01 00:00:00,-29.916,-71.2,147.0,290.0,4.6,17.0,NaN,...,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,1,1.0,1987429,2019-02-01
2,854880,99999,2019-02-01 01:00:00,-29.916,-71.2,147.0,290.0,3.6,17.0,NaN,...,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,1,1.0,1987430,2019-02-01
3,854880,99999,2019-02-01 02:00:00,-29.916,-71.2,147.0,280.0,1.0,17.0,NaN,...,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,1,1.0,1987431,2019-02-01
4,854880,99999,2019-02-01 03:00:00,-29.917,-71.2,146.0,290.0,2.6,16.6,1013.3,...,NaN,NaN,LA FLORIDA,CI,854880-99999,2019,1,1.0,1987432,2019-02-01


## Time Recent Input

This function takes in a datetime.timedelta and returns a dataset containing the data from datetime.now()-timedelta() to datetime.now().

In [20]:
tsd2 = tsd.time_recent(timedelta(weeks=4, days=0))
tsd2.to_pandas_dataframe().head(5)

/anaconda/envs/azureml_py36/lib/python3.6/site-packages/azureml/dataprep/api/dataflow.py:641: UserWarning: Inconsistent or mixed schemas detected across partitions. Using alternate reader.
  warnings.warn('Inconsistent or mixed schemas detected across partitions. Using alternate reader.')


,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,precipDepth,snowDepth,stationName,countryOrRegion,p_k,year,day,version,__index_level_0__,coarse_time
0,720735,73805,2019-09-01 00:53:00,30.349,-85.788,21.0,350.0,1.5,26.7,1016.2,...,0.0,NaN,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805,2019,1,1.0,117400,2019-09-01
1,720735,73805,2019-09-01 01:53:00,30.349,-85.788,21.0,20.0,3.1,26.7,1017.1,...,0.0,NaN,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805,2019,1,1.0,117401,2019-09-01
2,720735,73805,2019-09-01 02:53:00,30.349,-85.788,21.0,50.0,4.6,26.7,1017.4,...,0.0,NaN,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805,2019,1,1.0,117402,2019-09-01
3,720735,73805,2019-09-01 03:53:00,30.349,-85.788,21.0,40.0,3.1,25.6,1017.3,...,0.0,NaN,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805,2019,1,1.0,117403,2019-09-01
4,720735,73805,2019-09-01 04:53:00,30.349,-85.788,21.0,40.0,4.1,25.0,1017.1,...,0.0,NaN,NORTHWEST FLORIDA BEACHES INTL ARPT,US,720735-73805,2019,1,1.0,117404,2019-09-01


**NOTE:** This will return an empty dataframe there is no data within the last 2 days.

In [21]:
tsd2 = tsd.time_recent(timedelta(days=2))
tsd2.to_pandas_dataframe().tail(5)

""


### Drop Columns

<font color=red>The columns to be dropped should NOT include timstamp columns.</font><br>Below operation will lead to exception.

In [22]:
try:
    tsd2 = tsd.drop_columns(columns=['snowDepth', 'version', 'datetime'])
except Exception as e:
    print('Expected exception : {}'.format(str(e)))

Expected exception : Column datetime should not be dropped because it's a timestamp column. Please exclude it or use with_timestamp_columns with value None to reset it before dropping columns.


Drop will succeed if modify column list to exclude timestamp columns.

In [23]:
tsd2 = tsd.drop_columns(columns=['snowDepth', 'version', 'upload_date'])
tsd2.take(5).to_pandas_dataframe().sort_values(by='datetime')

,usaf,wban,datetime,latitude,longitude,elevation,windAngle,windSpeed,temperature,seaLvlPressure,...,pastWeatherIndicator,precipTime,precipDepth,stationName,countryOrRegion,p_k,year,day,__index_level_0__,coarse_time
0,722108,12894,2019-01-01 00:53:00,26.536,-81.755,9.0,150.0,3.1,22.8,1020.4,...,False,1.0,0.0,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,2161811,2019-01-01
1,722108,12894,2019-01-01 01:53:00,26.536,-81.755,9.0,130.0,2.1,21.7,1020.3,...,False,1.0,0.0,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,2161812,2019-01-01
2,722108,12894,2019-01-01 02:53:00,26.536,-81.755,9.0,100.0,2.6,20.0,1020.6,...,False,1.0,0.0,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,2161813,2019-01-01
3,722108,12894,2019-01-01 03:53:00,26.536,-81.755,9.0,110.0,1.5,18.9,1020.5,...,False,1.0,0.0,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,2161814,2019-01-01
4,722108,12894,2019-01-01 04:53:00,26.536,-81.755,9.0,0.0,0.0,18.9,1020.6,...,False,1.0,0.0,SW FLORIDA INTN AIRPORT,US,722108-12894,2019,1,2161815,2019-01-01


### Keep Columns

<font color=red>The columns to be kept should ALWAYS include timstamp columns.</font><br>Below operation will lead to exception.

In [24]:
try:
    tsd2 = tsd.keep_columns(columns=['snowDepth'], validate=False)
except Exception as e:
    print('Expected exception : {}'.format(str(e)))

Expected exception : Column datetime should be included because it's a timestamp column


Keep will succeed if modify column list to include timestamp columns.

In [25]:
tsd2 = tsd.keep_columns(columns=['snowDepth', 'datetime', 'coarse_time'], validate=False)
tsd2.to_pandas_dataframe().tail()

,datetime,snowDepth,coarse_time
31345,2019-09-02 14:00:00,NaN,2019-09-01
31346,2019-09-02 15:00:00,NaN,2019-09-01
31347,2019-09-02 16:00:00,NaN,2019-09-01
31348,2019-09-02 17:00:00,NaN,2019-09-01
31349,2019-09-02 18:00:00,NaN,2019-09-01


## Resetting Timestamp Columns

Rules for reseting are:
- You cannot assign 'None' to fine_grain_timestamp while assign a valid column name to coarse_grain_timestamp because coarse_grain_timestamp is optional while fine_grain_timestamp is mandatory for Tabular time series data.
- If you assign 'None' to fine_grain_timestamp, then both fine_grain_timestamp and coarse_grain_timestamp will all be cleared.
- If you assign only 'None' to coarse_grain_timestamp, then only coarse_grain_timestamp will be cleared.

In [26]:
# Illegal clearing, exception is expected.
try:
    tsd2 = tsd.with_timestamp_columns(fine_grain_timestamp=None, coarse_grain_timestamp='partition_date')
except Exception as e:
    print('Cleaning not allowed because {}'.format(str(e)))

# clear both
tsd2 = tsd.with_timestamp_columns(fine_grain_timestamp=None, coarse_grain_timestamp=None)
print('after clean both with None/None, timestamp columns are: {}'.format(tsd2.timestamp_columns))

# clear coarse_grain_timestamp only and assign 'datetime' as fine timestamp column
tsd2 = tsd2.with_timestamp_columns(fine_grain_timestamp='datetime', coarse_grain_timestamp=None)
print('after clean coarse timestamp column, timestamp columns are: {}'.format(tsd2.timestamp_columns))

Cleaning not allowed because Cannot assign coarse_grain_timestamp separately while fine_grain_timestamp is None.
after clean both with None/None, timestamp columns are: (None, None)
after clean coarse timestamp column, timestamp columns are: ('datetime', None)


In [ ]:
help(azureml.data.tabular_dataset)